In [3]:
import numpy as np
import pandas as pd


In [25]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task2.csv", dtype=dtype_map)
gaze_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_onoff_task2.csv", dtype=dtype_map)

In [ ]:

TICK_HZ = 10_000_000  # TIMETICK(f=10000000)
EPS = 1e-9

# --- Build ONE shared anchor per (Video, User) across any number of tables ---
def build_shared_anchor(*dfs,
                        tick_col: str = "TIMETICK(f=10000000)",
                        group_cols=("Video Name (folder)", "User ID (file)")) -> pd.DataFrame:
    refs = []
    for d in dfs:
        tmp = d[list(group_cols) + [tick_col]].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(tmp.groupby(list(group_cols), as_index=False)[tick_col].min()
                      .rename(columns={tick_col: "min_tick"}))
    ref = (pd.concat(refs, ignore_index=True)
             .groupby(list(group_cols), as_index=False)["min_tick"].min()
             .rename(columns={"min_tick": "anchor_tick"}))
    return ref

# 1) Window ids by TIMETICK (per Video+User), no task-start anchoring
def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec: float,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,
    out_col: str = "win_id",
) -> pd.DataFrame:
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

# 1b) Refine window labels by majority-fraction (label denoising)
def refine_window_labels_by_fraction(
    df_with_win: pd.DataFrame,
    *,
    group_cols=("Video Name (folder)", "User ID (file)"),
    win_col="win_id",
    label_col="label",
    on_hi=0.80,
    off_lo=0.20,
    out_col="label",
) -> pd.DataFrame:
    d = df_with_win.copy()
    d[label_col] = d[label_col].astype(str).str.upper().str.strip()
    d = d[d[label_col].isin(["ON","OFF"])].copy()

    key = list(group_cols) + [win_col]
    d["_is_on"] = (d[label_col] == "ON").astype(float)

    frac = (d.groupby(key, dropna=False)["_is_on"]
              .mean()
              .rename("on_fraction")
              .reset_index())

    frac["label_refined"] = np.where(frac["on_fraction"] >= on_hi, "ON",
                              np.where(frac["on_fraction"] <= off_lo, "OFF", np.nan))

    d = d.merge(frac[key + ["label_refined","on_fraction"]],
                on=key, how="left", validate="many_to_one")

    d = d[~d["label_refined"].isna()].copy()
    d[out_col] = d["label_refined"]
    d.drop(columns=["_is_on","label_refined"], inplace=True)
    return d

# ---- helpers ----
def _mad(x):
    x = pd.Series(x, dtype="float64").dropna()
    if x.empty: return np.nan
    med = x.median()
    return (x - med).abs().median()

def _pupil_slope(tick, pupil):
    # robust slope via np.polyfit on valid points; time in seconds from first sample
    if tick is None:
        return np.nan
    tt = pd.Series(tick, dtype="float64")
    pp = pd.Series(pupil, dtype="float64")
    m = (~tt.isna()) & (~pp.isna())
    if m.sum() < 5:
        return np.nan
    tsec = (tt[m] - tt[m].iloc[0]) / TICK_HZ
    try:
        slope, _ = np.polyfit(tsec.values, pp[m].values, 1)
    except Exception:
        slope = np.nan
    return slope

def _circular_variance_deg(angles_deg):
    a = pd.Series(angles_deg, dtype="float64").dropna()
    if a.empty: return np.nan
    rad = np.deg2rad(a.values)
    C = np.mean(np.cos(rad))
    S = np.mean(np.sin(rad))
    R = np.sqrt(C*C + S*S)
    return 1 - R  # 0=tightly clustered, 1=uniform

def _count_sign_changes(arr: np.ndarray) -> int:
    """Count sign flips in a 1D series (ignoring zeros)."""
    if arr.size < 3:
        return 0
    s = np.sign(arr.astype(float))
    # carry-forward to remove zeros
    for i in range(1, s.size):
        if s[i] == 0:
            s[i] = s[i-1]
    s = s[s != 0]
    if s.size < 2:
        return 0
    return int(np.sum(s[1:] * s[:-1] < 0))

def _shannon_entropy(p_vec: np.ndarray) -> float:
    p = np.asarray(p_vec, float)
    p = p[np.isfinite(p)]
    p = p[p > 0]
    if p.size == 0:
        return np.nan
    p = p / p.sum()
    return float(-(p * np.log2(p)).sum())

def _aoi_bucket(x):
    """Map raw AOI strings to coarse buckets TEXT/MAP/OTHER."""
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if "text" in s or "paragraph" in s or "title" in s:
        return "TEXT"
    if "map" in s or "legend" in s or "marker" in s:
        return "MAP"
    return "OTHER"

# ===============================================================
# Task-2: single windowing function (core + strong additions)
# ===============================================================
def window_features_task2(
    df: pd.DataFrame,
    *,
    win_sec: float = 3.0,
    min_valid_frac: float = 0.60,
    left_x_thresh: float = 0.45,
    right_x_thresh: float = 0.55,
) -> pd.DataFrame:
    """
    Computes Task-2 core features + strong additions in one pass.
    Retains: Task Id, delay, ease, direction, type, label, phase, block_id, block_anim.

    Assumes df already has a 'win_id' column (use add_window_id_by_timetick first).
    """
    # columns to carry through
    carry_cols = ["Task Id","delay","ease","direction","type","label","phase","block_id","block_anim"]
    base_keys  = ["Video Name (folder)","User ID (file)","win_id"] + carry_cols

    need_cols = {"FPOGX","FPOGY","FPOGV"}
    has_coords = need_cols.issubset(df.columns)
    has_aoi    = "AOI" in df.columns

    tick_col = "TIMETICK(f=10000000)"

    def _agg(w: pd.DataFrame) -> pd.Series:
        out = {}

        # -------- carry-through (first non-null in window) --------
        for c in carry_cols:
            out[c] = w[c].dropna().iloc[0] if c in w and w[c].notna().any() else (w[c].iloc[0] if c in w else np.nan)

        # -------- validity & basic masks --------
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        valid_frac = float(gv.mean()) if len(w) else 0.0
        out["valid_gaze_ratio"] = valid_frac

        # =======================
        # CORE: Fixation features
        # =======================
        if "FPOGID" in w:
            fix_count = w["FPOGID"].dropna().nunique()
        else:
            fix_count = np.nan
        out["fix_rate_per_s"]  = (fix_count / win_sec) if pd.notna(fix_count) else np.nan
        out["fix_median_dur"]  = (w["FPOGD"].median() if "FPOGD" in w else np.nan)

        # =====================
        # CORE: Gaze dispersion
        # =====================
        if has_coords:
            gx = w.loc[gv, "FPOGX"].std()
            gy = w.loc[gv, "FPOGY"].std()
            out["gaze_dispersion_total"] = (
                np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2))
                if (not pd.isna(gx) or not pd.isna(gy)) else np.nan
            )
        else:
            out["gaze_dispersion_total"] = np.nan

        # ==================
        # CORE: Saccade info
        # ==================
        if "SACCADE_MAG" in w and gv.any():
            mags = w.loc[gv, "SACCADE_MAG"].dropna().to_numpy()
            out["saccade_mean_mag"] = float(np.mean(mags)) if mags.size else np.nan
            # saccade_rate_per_s: prefer event id; else rising edges
            if "SACCADE_ID" in w:
                sacc_count = int(w.loc[gv, "SACCADE_ID"].dropna().nunique())
            else:
                mag = w.loc[gv, "SACCADE_MAG"].fillna(0.0).to_numpy()
                rising = (mag[1:] > 0) & (mag[:-1] <= 0) if mag.size >= 2 else np.array([], dtype=bool)
                sacc_count = int(rising.sum())
            out["saccade_rate_per_s"] = sacc_count / max(win_sec, EPS)
        else:
            out["saccade_mean_mag"] = np.nan
            out["saccade_rate_per_s"] = np.nan

        # =============================
        # CORE: dir_change_rate, axis_ratio
        # =============================
        if has_coords and valid_frac >= min_valid_frac:
            x = w.loc[gv, "FPOGX"].to_numpy(float)
            y = w.loc[gv, "FPOGY"].to_numpy(float)
            sx, sy = float(np.nanstd(x)), float(np.nanstd(y))
            out["axis_ratio"] = (sy / (sx + EPS)) if np.isfinite(sx) and np.isfinite(sy) else np.nan
            v = np.diff(y) if np.nan_to_num(sy) >= np.nan_to_num(sx) else np.diff(x)
            out["dir_change_rate"] = _count_sign_changes(v) / max(win_sec, EPS) if v.size else np.nan
        else:
            out["axis_ratio"] = np.nan
            out["dir_change_rate"] = np.nan

        # =================================
        # STRONG additions: AOI-aware block
        # =================================
        out["dwell_text_ratio"] = np.nan
        out["dwell_map_ratio"]  = np.nan
        out["aoi_entropy"]      = np.nan
        out["trans_text_to_map"] = np.nan
        out["trans_map_to_text"] = np.nan
        out["TTFF_text"]        = np.nan

        # Fallback (X-split) fields — always create
        out["x_mean"]            = np.nan
        out["x_dir_change_rate"] = np.nan
        out["left_ratio"]        = np.nan
        out["right_ratio"]       = np.nan
        out["x_entropy"]         = np.nan

        # AOI-first if present & enough valid gaze
        if has_aoi and valid_frac >= min_valid_frac:
            aoi_bucket = w["AOI"].map(_aoi_bucket)
            aoi_bucket = aoi_bucket.where(gv, other=np.nan)

            n_valid = int((~aoi_bucket.isna()).sum())
            if n_valid > 0:
                text_mask  = aoi_bucket.eq("TEXT")
                map_mask   = aoi_bucket.eq("MAP")
                other_mask = aoi_bucket.eq("OTHER")
                dt = float(text_mask.sum()) / n_valid
                dm = float(map_mask.sum())  / n_valid
                do = float(other_mask.sum())/ n_valid
                out["dwell_text_ratio"] = dt
                out["dwell_map_ratio"]  = dm
                out["aoi_entropy"]      = _shannon_entropy(np.array([dt, dm, do]))

                seq = aoi_bucket.dropna().to_numpy(str)
                if seq.size >= 2:
                    t2m = m2t = 0
                    for a, b in zip(seq[:-1], seq[1:]):
                        if a != b:
                            if   a == "TEXT" and b == "MAP":  t2m += 1
                            elif a == "MAP"  and b == "TEXT": m2t += 1
                    out["trans_text_to_map"] = t2m / max(win_sec, EPS)
                    out["trans_map_to_text"] = m2t / max(win_sec, EPS)

                if tick_col in w.columns:
                    try:
                        tick = pd.to_numeric(w[tick_col], errors="coerce")
                        tick0 = tick.iloc[0]
                        idx_first_text = text_mask[text_mask].index[0]
                        ttff = float((tick.loc[idx_first_text] - tick0) / TICK_HZ)
                        if ttff < 0 or not np.isfinite(ttff): ttff = np.nan
                        out["TTFF_text"] = ttff
                    except Exception:
                        out["TTFF_text"] = np.nan

        # Fallback (or complementary) X-split features if coords OK
        if has_coords and valid_frac >= min_valid_frac:
            x = w.loc[gv, "FPOGX"].to_numpy(float)
            out["x_mean"] = float(np.nanmean(x)) if x.size else np.nan
            left_mask  = x < left_x_thresh
            right_mask = x > right_x_thresh
            left_ratio  = float(np.mean(left_mask))  if x.size else np.nan
            right_ratio = float(np.mean(right_mask)) if x.size else np.nan
            mid_ratio = (1.0 - (left_ratio + right_ratio)) if np.all(np.isfinite([left_ratio,right_ratio])) else np.nan
            out["left_ratio"]  = left_ratio
            out["right_ratio"] = right_ratio
            out["x_entropy"]   = _shannon_entropy(np.array([left_ratio, mid_ratio, right_ratio])) if np.isfinite(mid_ratio) else np.nan
            dx = np.diff(x)
            out["x_dir_change_rate"] = _count_sign_changes(dx) / max(win_sec, EPS) if dx.size else np.nan

        # Optional: Pupil and direction variance (kept here for completeness)
        # (Not in your core/strong list, but easy to compute and harmless to keep)
        if "LPMM" in w and "RPMM" in w:
            lv = (w["LPMMV"] == 1) if "LPMMV" in w else pd.Series(False, index=w.index)
            rv = (w["RPMMV"] == 1) if "RPMMV" in w else pd.Series(False, index=w.index)
            l = w["LPMM"].where(lv)
            r = w["RPMM"].where(rv)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            out["pupil_mad"]   = _mad(pupil_row)
            out["pupil_slope"] = _pupil_slope(w[tick_col] if tick_col in w else None, pupil_row)
        else:
            out["pupil_mad"] = np.nan
            out["pupil_slope"] = np.nan

        if "SACCADE_DIR" in w and gv.any():
            out["saccade_dir_var"]  = w.loc[gv, "SACCADE_DIR"].var(ddof=1)
            out["saccade_circ_var"] = _circular_variance_deg(w.loc[gv, "SACCADE_DIR"])
        else:
            out["saccade_dir_var"]  = np.nan
            out["saccade_circ_var"] = np.nan

        return pd.Series(out)

    # Group and compute
    needed_for_group = ["Video Name (folder)","User ID (file)","win_id"] + [c for c in carry_cols if c in df.columns]
    d = (df.dropna(subset=["win_id"])
           .groupby(needed_for_group, dropna=False)
           .apply(_agg)
           .reset_index())

    # Enforce validity mask on layout features if you want to blank when gaze is low
    if "valid_gaze_ratio" in d.columns:
        bad = d["valid_gaze_ratio"] < min_valid_frac
        for c in [
            "saccade_rate_per_s","dir_change_rate","axis_ratio",
            "x_mean","x_dir_change_rate","left_ratio","right_ratio","x_entropy",
            "dwell_text_ratio","dwell_map_ratio","aoi_entropy",
            "trans_text_to_map","trans_map_to_text","TTFF_text"
        ]:
            if c in d.columns:
                d.loc[bad, c] = np.nan

    return d

In [42]:
# after you add win_id and refine labels for Task 2:
gaze_wid_t2 = add_window_id_by_timetick(gaze_onoff, win_sec=3.0)
# use the Task-2 gaze features:
gaz_w_t2 = window_features_gaze_task2(gaze_wid_t2, win_sec=3.0, min_valid_frac=0.60)
fix_wid_t2 = add_window_id_by_timetick(fixations_onoff, win_sec=3.0)
fix_w_t2 = window_features_fixations(fix_wid_t2, win_sec=3.0)
win_task2 = combine_fix_gaze_windows(fix_w_t2, gaz_w_t2)

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_6438/3941962550.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_6438/1676081714.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_extras_coord)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_6438/1676081714.py:166: DeprecationWarning: DataFrameGroupBy

In [43]:
win_task2.to_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs2/fix_wid_t2.csv", index=False)    